# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
**Rule in plain words:** A page is worth flagging for refresh-and-CTR-fix action if it still
gets meaningful search visibility, it hasn't been updated in a long time, and its CTR is
underperforming what similar-position pages typically get.

**Signal 1 — staleness (behind the refresh flags):** hypothesis — pages that haven't been
updated in longer get flagged as declining more often. Tested via `freshness_tier` bucketed
against `is_declining_label`. Verdict: [staleness verdict: MIXED. Decline rate rises with staleness through 0-30 (0.511, n=20,480) →
31-90 (0.589, n=175) → 91-180 (0.611, n=9,171), supporting the hypothesis — but reverses at 181+
(0.471, n=174). Both extreme tiers have thin samples (n<200), so the reversal likely reflects
noise rather than a real "too stale to matter" effect, but it prevents a clean CONFIRMED.


**Signal 2 — CTR-vs-position (behind the CTR-fix logic):** hypothesis — pages whose CTR falls
well below the median CTR for their position tier are more likely to be declining. Tested only
on `measurable_opportunity == 1` rows (volume floor). Verdict: CTR-vs-position verdict: CONFIRMED. CTR-underperforming pages decline at 66.6%
(n=8,736) vs 55.2% for non-underperformers (n=13,270) and a 54.2% base rate — a clear,
sizeable gap on a well-powered sample.

**Reason code:** `stale_and_ctr_underperforming`
**Action label:** `review_for_refresh_and_ctr_fix`



In [2]:
!git clone https://github.com/Prakritibhandari07/FlyRank-ml-internship.git
%cd FlyRank-ml-internship

Cloning into 'FlyRank-ml-internship'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 149 (delta 58), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.86 MiB | 9.43 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/FlyRank-ml-internship


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

base_rate = df["is_declining_label"].mean()
print(f"Base rate (n={len(df)}): {base_rate:.3f}")

signal_a = (
    df.groupby("freshness_tier", observed=True)
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .reindex(["0-30", "31-90", "91-180", "181+"])
)
print("\nSignal 1 — freshness_tier vs decline_rate")
print(signal_a)

df["measurable_opportunity"] = (
    (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)
).astype(int)
measurable = df[df["measurable_opportunity"] == 1].copy()

tier_median_ctr = measurable.groupby("position_tier", observed=True)["ctr"].transform("median")
measurable["ctr_underperform"] = (measurable["ctr"] < 0.7 * tier_median_ctr).astype(int)

signal_b = (
    measurable.groupby("ctr_underperform")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
)
print(f"\nSignal 2 — ctr_underperform vs decline_rate (n={len(measurable)})")
print(signal_b)

Base rate (n=30000): 0.542

Signal 1 — freshness_tier vs decline_rate
                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264

Signal 2 — ctr_underperform vs decline_rate (n=22006)
                      n  decline_rate
ctr_underperform                     
0                 13270      0.552449
1                  8736      0.666323


## 2. Build the ranked queue (writes the CSV)
Score is a transparent multiplicative rule — no fitted weights, every factor is a 0/1 gate
except `impressions_90d`, which breaks ties by visibility.


In [4]:
import os

stale = df["freshness_tier"].isin(["91-180", "181+"]).astype(int)
visible = df["measurable_opportunity"]

tier_median_ctr_full = df.groupby("position_tier", observed=True)["ctr"].transform("median")
ctr_underperform_full = np.where(
    df["measurable_opportunity"] == 1,
    (df["ctr"] < 0.7 * tier_median_ctr_full).astype(int),
    0
)

df["score"] = stale * visible * ctr_underperform_full * df["impressions_90d"]
df["reason_code"] = np.where(df["score"] > 0, "stale_and_ctr_underperforming", "")
df["action"] = np.where(df["score"] > 0, "review_for_refresh_and_ctr_fix", "no_action")

queue = (
    df[df["score"] > 0]
    .sort_values("score", ascending=False)
    .loc[:, ["content_id", "client_id", "score", "reason_code", "action",
             "freshness_tier", "days_since_last_update", "ctr", "position_tier",
             "avg_position", "impressions_90d", "search_volume"]]
)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue: {len(queue)} flagged rows out of {len(df)} total")
queue.head(10)

Ranked queue: 2629 flagged rows out of 30000 total


,content_id,client_id,score,reason_code,action,freshness_tier,days_since_last_update,ctr,position_tier,avg_position,impressions_90d,search_volume
3394,content_36ff89c8214e,client_19581e27de,295097,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.05,page_1,7.3,295097,0.0
7445,content_c8e9d6ab9013,client_19581e27de,208678,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.00,page_1,9.7,208678,20.0
26474,content_a7427266c305,client_19581e27de,201111,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.11,page_1,5.7,201111,0.0
3070,content_91652435f57a,client_19581e27de,159590,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.06,page_1,7.8,159590,10.0
5621,content_97a86caf3a3d,client_19581e27de,147670,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.07,page_1,6.4,147670,40.0
22694,content_8b36799b7e44,client_6208ef0f77,141400,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.02,page_3_5,32.0,141400,0.0
9193,content_c1fe78bc4e37,client_19581e27de,134055,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.03,page_1,7.5,134055,70.0
6689,content_e752a4e03dd3,client_6208ef0f77,130892,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.01,page_3_5,23.9,130892,0.0
3343,content_54baba704595,client_6208ef0f77,130617,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.01,page_3_5,47.0,130617,0.0
19183,content_124763d39ca5,client_6208ef0f77,129803,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.01,page_3_5,33.2,129803,0.0


## 3. Top-20 review
1. content_36ff89c8214e — review_for_refresh_and_ctr_fix — page_1 position (7.3) with massive
   volume (295,097 impr/90d) but CTR only 0.05% — a page 1 spot getting almost no clicks.
   Would be wrong if this page's snippet/title was recently changed and CTR hasn't caught up yet.

2. content_c8e9d6ab9013 — page_1 (9.7), 208,678 impressions, CTR 0.00% — zero clicks despite
   huge visibility. Would be wrong if this is a non-clickable result type (e.g. featured
   snippet cannibalization) rather than a genuine content problem.

3. content_a7427266c305 — page_1 (5.7), 201,111 impressions, CTR 0.11%, search_volume 0 —
   strong position, near-zero demand signal on the keyword itself. Would be wrong if
   search_volume=0 reflects a data gap rather than true zero demand.

4. content_91652435f57a — page_1 (7.8), 159,590 impressions, CTR 0.06% — same profile as #1.
   Would be wrong if the title tag is already optimized and the real issue is meta description,
   which this data can't see.

5. content_97a86caf3a3d — page_1 (6.4), 147,670 impressions, CTR 0.07%, volume 40 — solid
   position, thin CTR. Would be wrong if intent mismatch (informational query, transactional
   page) rather than a fixable CTR issue.

6. content_8b36799b7e44 — page_3_5 (32.0), 141,400 impressions, CTR 0.02% — position itself is
   the bigger problem here (page 3-5), not just CTR; refresh may need to target ranking, not
   just snippet. Would be wrong if flagged purely as a "CTR fix" when it's really a ranking gap.

7. content_c1fe78bc4e37 — page_1 (7.5), 134,055 impressions, CTR 0.03%, volume 70 — good
   demand signal, page 1, near-zero clicks. Would be wrong if recent SERP layout changes
   (ads, other features) are suppressing organic clicks regardless of content quality.

8. content_e752a4e03dd3 — page_3_5 (23.9), 130,892 impressions, CTR 0.01% — like #6, position
   is likely the dominant issue, not CTR alone. Would be wrong if treated as a simple
   snippet-rewrite fix.

9. content_54baba704595 — page_3_5 (47.0), 130,617 impressions, CTR 0.01% — deep into page 3-5,
   borderline page 4; CTR here is nearly irrelevant since visibility itself is poor. Would be
   wrong if this is flagged as a "quick CTR fix" when it actually needs a ranking-focused refresh.

10. content_124763d39ca5 — page_3_5 (33.2), 129,803 impressions, CTR 0.01% — same pattern as #9.
    Would be wrong for the same reason: position, not snippet copy, is the likely root cause.

11. content_4c76e9b13aea — page_1 (7.4), 127,952 impressions, CTR 0.07%, volume 10 — solid
    position, near-zero demand on the mapped keyword. Would be wrong if the keyword mapping
    itself is stale and the page actually ranks for different, higher-demand terms not captured
    here.

12. content_b115f7c74779 — page_1 (8.0), 123,469 impressions, CTR 0.03% — consistent with the
    page_1-low-CTR pattern seen across this client. Would be wrong if this specific page recently
    had a positive event (e.g. featured in a newsletter) inflating impressions temporarily.

13. content_647e177596e9 — page_1 (7.4), 111,222 impressions, CTR 0.09%, volume 70 — best CTR
    among the page_1 rows here, still low. Would be wrong if 0.09% is actually reasonable for
    this content_type/intent and the tier median comparison isn't apples-to-apples.

14. content_42d423551e2c — page_1 (4.8), 106,652 impressions, CTR 0.09% — strongest position in
    the top 20 (avg 4.8) yet still underperforming CTR. Would be wrong if this is a branded query
    where low CTR is structurally normal (users recognize the brand but don't click through).

15. content_5d3dfb80a423 — page_1 (6.5), 97,235 impressions, CTR 0.07%, volume 140 — highest
    search_volume in the top 20, meaningful real demand behind this keyword. Strong candidate
    regardless of other caveats. Would be wrong if seasonal demand inflated volume temporarily.

16. content_867dc005867d — page_1 (6.8), 95,522 impressions, CTR 0.10% — highest CTR among the
    page_1 group, borderline whether it's truly "underperforming" or just barely below threshold.
    Would be wrong if the 0.7× tier-median cutoff is too aggressive for this specific case.

17. content_87dfc063bf4e — page_1 (4.5), 90,991 impressions, CTR 0.08% — strong position, still
    flagged. Would be wrong if this page's real issue is seasonality rather than staleness.

18. content_f8de7d4cee60 — page_3_5 (30.8), 89,803 impressions, CTR 0.02% — position-driven
    issue again, like #6/#8/#9/#10. Would be wrong if treated as primarily a CTR problem.

19. content_32cfb0b2fccf — page_3_5 (38.5), 89,361 impressions, CTR 0.01% — same pattern,
    deep page 3-5 position dominates the story over CTR. Would be wrong for the same reason.

20. content_fb4bf6555c79 — page_3_5 (45.6), 84,093 impressions, CTR 0.00% — weakest position and
    zero clicks in the top 20; likely the least "CTR-fixable" of the batch. Would be wrong if
    flagged as a snippet-rewrite candidate when ranking is clearly the dominant problem.


In [5]:
top20 = queue.head(20).reset_index(drop=True)
top20

,content_id,client_id,score,reason_code,action,freshness_tier,days_since_last_update,ctr,position_tier,avg_position,impressions_90d,search_volume
0,content_36ff89c8214e,client_19581e27de,295097,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.05,page_1,7.3,295097,0.0
1,content_c8e9d6ab9013,client_19581e27de,208678,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.00,page_1,9.7,208678,20.0
2,content_a7427266c305,client_19581e27de,201111,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.11,page_1,5.7,201111,0.0
3,content_91652435f57a,client_19581e27de,159590,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.06,page_1,7.8,159590,10.0
4,content_97a86caf3a3d,client_19581e27de,147670,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.07,page_1,6.4,147670,40.0
5,content_8b36799b7e44,client_6208ef0f77,141400,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.02,page_3_5,32.0,141400,0.0
6,content_c1fe78bc4e37,client_19581e27de,134055,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.03,page_1,7.5,134055,70.0
7,content_e752a4e03dd3,client_6208ef0f77,130892,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.01,page_3_5,23.9,130892,0.0
8,content_54baba704595,client_6208ef0f77,130617,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.01,page_3_5,47.0,130617,0.0
9,content_124763d39ca5,client_6208ef0f77,129803,stale_and_ctr_underperforming,review_for_refresh_and_ctr_fix,91-180,104,0.01,page_3_5,33.2,129803,0.0


## 4. Weak picks + leakage check
Weak picks found: the automated filter (missing search_volume, no_data position tier, or
impressions < 200) returned 0 — every row in the top 20 has substantial volume and complete
data, so no row is weak on a *data-quality* basis.

But that's the wrong lens alone — looking harder by eye, 7 of the 20 rows (content_8b36799b7e44,
content_e752a4e03dd3, content_54baba704595, content_124763d39ca5, content_f8de7d4cee60,
content_32cfb0b2fccf, content_fb4bf6555c79) all share position_tier = page_3_5, with avg_position
ranging 23.9–47.0. These are genuinely weak picks for the specific action label the rule assigns:
the reason code says `ctr_underperforming` and the action says `review_for_refresh_and_ctr_fix`,
but a page ranking at position 30-47 has a structural CTR ceiling — organic CTR at that depth is
near-zero regardless of snippet quality (real-world CTR curves drop to near 0 past position ~20).
Flagging these for a "CTR fix" is likely the wrong intervention; they need a ranking-focused
refresh, not a snippet/title rewrite. The rule's single reason code can't distinguish "page 1,
bad snippet" from "page 3-5, snippet is irrelevant because nobody scrolls that far" — that's a
real weakness in a one-reason-code design, not a data problem the automated weak-pick filter
was built to catch.

Leakage: confirmed NONE — the score is built only from freshness_tier, measurable_opportunity,
position_tier, ctr, and impressions_90d. No label-derived (trend_direction, trend_pct,
is_declining_label) or future-window (impressions_last_30d, impressions_prev_30d) columns were
used anywhere in the rule.

In [6]:
used_in_rule = ["freshness_tier", "measurable_opportunity", "position_tier", "ctr", "impressions_90d"]
forbidden = ["trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "impressions_prev_30d"]
leak_hits = [c for c in used_in_rule if c in forbidden]
print("Leakage check:", leak_hits if leak_hits else "NONE — clean")

weak = top20[
    (top20["search_volume"].isna()) |
    (top20["position_tier"] == "no_data") |
    (top20["impressions_90d"] < 200)
]
print(f"\nWeak picks in top 20: {len(weak)}")
weak

Leakage check: NONE — clean

Weak picks in top 20: 0


,content_id,client_id,score,reason_code,action,freshness_tier,days_since_last_update,ctr,position_tier,avg_position,impressions_90d,search_volume


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.